# EchoMind — Prosody v5 — Resume Cache-Only

Este notebook serve para **continuar a partir da pasta `prosody_outputs_v5` no Google Drive**, sem voltar a carregar WavLM e sem voltar a extrair embeddings SSL.

Objetivo: carregar os ficheiros já gerados, reconstruir `X_manual`, `X_ssl_aligned`, `y`, `groups` e `sources`, correr benchmark e guardar os resultados.

**Não usa `transformers`, `torch` nem `AutoModel`.**

In [1]:
# 0) Reparação automática do ambiente — corre uma vez e reinicia o runtime.
# Isto evita erros do tipo:
# ValueError: numpy.dtype size changed
# ImportError: cannot import name '_center' from numpy._core.umath

AUTO_REPAIR_ENV = True  # deixa True se tinhas erro de numpy/scipy/sklearn/torch

import sys, os, subprocess
from pathlib import Path

FLAG = Path("/content/.prosody_resume_env_repaired")

if AUTO_REPAIR_ENV and not FLAG.exists():
    print("A reparar ambiente científico: numpy / scipy / scikit-learn / pandas...")
    pkgs = [
        "numpy==1.26.4",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "pandas==2.2.2",
        "joblib==1.4.2",
        "matplotlib==3.8.4",
        "threadpoolctl==3.5.0",
    ]
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "--force-reinstall", "-q"
    ] + pkgs)
    FLAG.write_text("done", encoding="utf-8")
    print("\n✅ Ambiente reparado. Vou reiniciar o runtime agora.")
    print("Depois do reinício, corre o notebook novamente a partir da próxima célula.")
    os.kill(os.getpid(), 9)
else:
    print("Ambiente já reparado nesta sessão, ou AUTO_REPAIR_ENV=False. Podes continuar.")

A reparar ambiente científico: numpy / scipy / scikit-learn / pandas...


: 

: 

: 

## 1. Imports, Drive e configuração

Esta parte só carrega bibliotecas normais de análise e modelos clássicos. Não há WavLM aqui.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
import sklearn
print("sklearn:", sklearn.__version__)

In [ ]:
# Configuração principal

OUTPUT_DIR = Path("/content/drive/MyDrive/prosody_outputs_v5")

EMOTIONS = ["joy", "sadness", "surprise", "fear", "anger", "disgust", "neutral"]

RANDOM_STATE = 42
N_SPLITS_GROUP_CV = 5

# Para correr mais depressa, mete RUN_EXTRA_TREES=False.
# Para avaliação mais forte, deixa True.
RUN_EXTRA_TREES = True
EXTRA_TREES_N_ESTIMATORS = 300

# Leave-One-Source-Out é útil, mas pode demorar. Deixa False se só queres benchmark rápido.
RUN_LEAVE_ONE_SOURCE_OUT = False

print("OUTPUT_DIR:", OUTPUT_DIR)
print("Existe?", OUTPUT_DIR.exists())

## 2. Verificar se a cache existe

Estes são os ficheiros essenciais que permitem continuar sem voltar a extrair WavLM.

In [ ]:
required_files = {
    "manual_features": OUTPUT_DIR / "manual_features_v5.csv",
    "manual_feature_columns": OUTPUT_DIR / "manual_feature_columns_v5.json",
    "ssl_embeddings": OUTPUT_DIR / "ssl_embeddings_v5_wavlm_base.npz",
    "ssl_meta": OUTPUT_DIR / "ssl_embeddings_v5_wavlm_base_meta.csv",
}

optional_files = {
    "metadata": OUTPUT_DIR / "metadata_v5_clean.csv",
    "ssl_feature_columns": OUTPUT_DIR / "ssl_feature_columns_v5.json",
}

print("Ficheiros obrigatórios:")
for name, path in required_files.items():
    exists = path.exists()
    size = round(path.stat().st_size / 1024 / 1024, 2) if exists else None
    print(f"{name:<24} {str(exists):<5} {str(size):>8} MB  {path.name}")

missing = [str(p) for p in required_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Faltam ficheiros obrigatórios na pasta prosody_outputs_v5:\n" + "\n".join(missing)
    )

print("\nFicheiros opcionais:")
for name, path in optional_files.items():
    exists = path.exists()
    size = round(path.stat().st_size / 1024 / 1024, 2) if exists else None
    print(f"{name:<24} {str(exists):<5} {str(size):>8} MB  {path.name}")

## 3. Carregar features manuais, metadata e embeddings SSL

O alinhamento tenta primeiro usar `path`. Se os caminhos absolutos mudaram, usa uma chave relativa do tipo `Angry/ficheiro.wav`.

In [ ]:
FOLDER_LABEL_MAP = {
    "Angry": "anger",
    "Happy": "joy",
    "Sad": "sadness",
    "Neutral": "neutral",
    "Fearful": "fear",
    "Disgusted": "disgust",
    "Surprised": "surprise",
    "Suprised": "surprise",
    "angry": "anger",
    "happy": "joy",
    "sad": "sadness",
    "neutral": "neutral",
    "fearful": "fear",
    "disgusted": "disgust",
    "surprised": "surprise",
    "suprised": "surprise",
}

def load_json_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def make_rel_key_from_path(path_value):
    """
    Cria uma chave estável tipo 'Angry/ficheiro.wav',
    mesmo que o caminho absoluto tenha mudado.
    """
    p = Path(str(path_value))
    parts = list(p.parts)

    if "Emotions" in parts:
        i = parts.index("Emotions")
        if len(parts) > i + 2:
            return f"{parts[i+1]}/{p.name}"

    return f"{p.parent.name}/{p.name}"

def add_rel_key(df):
    df = df.copy()

    if "path" in df.columns:
        df["path"] = df["path"].astype(str)
        df["rel_key"] = df["path"].apply(make_rel_key_from_path)
    elif {"folder", "filename"}.issubset(df.columns):
        df["rel_key"] = df["folder"].astype(str) + "/" + df["filename"].astype(str)
    elif "filename" in df.columns:
        raise RuntimeError("Tenho filename mas não tenho path/folder para criar rel_key.")
    else:
        raise RuntimeError("Não encontro colunas suficientes para criar rel_key.")

    return df

def show_basic_info(name, df):
    print(f"\n{name}: {df.shape}")
    print("Colunas:", list(df.columns[:12]), "..." if len(df.columns) > 12 else "")

manual_path = required_files["manual_features"]
manual_cols_path = required_files["manual_feature_columns"]
ssl_npz_path = required_files["ssl_embeddings"]
ssl_meta_path = required_files["ssl_meta"]
metadata_path = optional_files["metadata"]

manual_feature_cols = load_json_list(manual_cols_path)
manual_df = pd.read_csv(manual_path)
manual_df = add_rel_key(manual_df)

ssl_meta = pd.read_csv(ssl_meta_path)
ssl_meta = add_rel_key(ssl_meta)
ssl_meta = ssl_meta.reset_index(drop=True)
ssl_meta["ssl_row"] = np.arange(len(ssl_meta))

z = np.load(ssl_npz_path, allow_pickle=True)
print("NPZ keys:", z.files)

if "X_ssl" in z.files:
    X_ssl = z["X_ssl"]
elif "embeddings" in z.files:
    X_ssl = z["embeddings"]
elif "X" in z.files:
    X_ssl = z["X"]
elif "arr_0" in z.files:
    X_ssl = z["arr_0"]
else:
    raise RuntimeError(f"Não encontrei matriz de embeddings no npz. Keys: {z.files}")

X_ssl = X_ssl.astype(np.float32)

show_basic_info("manual_df", manual_df)
show_basic_info("ssl_meta", ssl_meta)
print("\nmanual_feature_cols:", len(manual_feature_cols))
print("X_ssl:", X_ssl.shape)

if len(ssl_meta) != X_ssl.shape[0]:
    raise RuntimeError(
        f"ssl_meta tem {len(ssl_meta)} linhas, mas X_ssl tem {X_ssl.shape[0]} embeddings."
    )

## 4. Reconstruir `aligned_df`, `X_manual`, `X_ssl_aligned`, `y`, `groups` e `sources`

In [ ]:
needed_meta_cols = {"label", "speaker", "source_dataset"}

if needed_meta_cols.issubset(set(manual_df.columns)):
    base_df = manual_df.copy()
    print("✅ A usar label/speaker/source_dataset diretamente de manual_features_v5.csv")
else:
    print("manual_features não tem metadata completo. Vou tentar juntar com metadata_v5_clean.csv.")
    if not metadata_path.exists():
        raise FileNotFoundError(
            "manual_features não tem metadata completo e metadata_v5_clean.csv não existe."
        )

    metadata = pd.read_csv(metadata_path)
    metadata = add_rel_key(metadata)
    print("metadata:", metadata.shape)

    base_df = metadata[["rel_key", "label", "speaker", "source_dataset"]].merge(
        manual_df,
        on="rel_key",
        how="inner"
    )

print("base_df inicial:", base_df.shape)

# Limpeza defensiva
base_df = base_df.copy()
base_df["label"] = base_df["label"].astype(str)
base_df["speaker"] = base_df["speaker"].astype(str)
base_df["source_dataset"] = base_df["source_dataset"].astype(str)

base_df = base_df[base_df["label"].isin(EMOTIONS)].copy()
base_df = base_df[~base_df["source_dataset"].str.lower().eq("unknown")].copy()
base_df = base_df[~base_df["speaker"].str.lower().eq("unknown")].copy()

print("base_df após limpeza:", base_df.shape)

# Escolher chave de alinhamento: path se possível; senão rel_key.
path_overlap = 0
if "path" in base_df.columns and "path" in ssl_meta.columns:
    path_overlap = len(set(base_df["path"].astype(str)).intersection(set(ssl_meta["path"].astype(str))))

rel_overlap = len(set(base_df["rel_key"].astype(str)).intersection(set(ssl_meta["rel_key"].astype(str))))

print("overlap por path:", path_overlap)
print("overlap por rel_key:", rel_overlap)

join_key = "path" if path_overlap > 0 else "rel_key"
print("Chave escolhida para alinhamento:", join_key)

# Evitar explosão por duplicados na chave
dup_base = int(base_df.duplicated(join_key).sum())
dup_ssl = int(ssl_meta.duplicated(join_key).sum())

if dup_base:
    print(f"⚠️ base_df tem {dup_base} duplicados em {join_key}. Vou manter a primeira ocorrência.")
    base_df = base_df.drop_duplicates(join_key, keep="first")

if dup_ssl:
    print(f"⚠️ ssl_meta tem {dup_ssl} duplicados em {join_key}. Vou manter a primeira ocorrência.")
    ssl_meta = ssl_meta.drop_duplicates(join_key, keep="first")

aligned_df = base_df.merge(
    ssl_meta[[join_key, "ssl_row"]],
    on=join_key,
    how="inner"
)

print("\naligned_df:", aligned_df.shape)

if aligned_df.empty and join_key == "path":
    print("⚠️ Alinhamento por path deu vazio. Vou tentar por rel_key.")
    aligned_df = base_df.merge(
        ssl_meta[["rel_key", "ssl_row"]],
        on="rel_key",
        how="inner"
    )
    print("aligned_df por rel_key:", aligned_df.shape)

if aligned_df.empty:
    raise RuntimeError("Não consegui alinhar manual_features com ssl_embeddings.")

missing_manual_cols = [c for c in manual_feature_cols if c not in aligned_df.columns]
if missing_manual_cols:
    raise RuntimeError(f"Faltam colunas manuais. Exemplos: {missing_manual_cols[:20]}")

X_manual_df = aligned_df[manual_feature_cols].apply(pd.to_numeric, errors="coerce")
X_manual_df = X_manual_df.replace([np.inf, -np.inf], np.nan)

X_manual = X_manual_df.to_numpy(dtype=np.float32)
X_ssl_aligned = X_ssl[aligned_df["ssl_row"].to_numpy()].astype(np.float32)
X_ssl_aligned = np.nan_to_num(X_ssl_aligned, nan=0.0, posinf=0.0, neginf=0.0)

y = aligned_df["label"].astype(str).to_numpy()
groups = aligned_df["speaker"].astype(str).to_numpy()
sources = aligned_df["source_dataset"].astype(str).to_numpy()

class_priors = {e: float(np.mean(y == e)) for e in EMOTIONS}
with open(OUTPUT_DIR / "class_priors_v5_resume.json", "w", encoding="utf-8") as f:
    json.dump(class_priors, f, ensure_ascii=False, indent=2)

print("\n✅ Matrizes reconstruídas:")
print("X_manual:", X_manual.shape)
print("X_ssl_aligned:", X_ssl_aligned.shape)
print("y:", y.shape)
print("groups:", groups.shape, "| n_unique:", len(np.unique(groups)))
print("sources:", sources.shape, "| n_unique:", len(np.unique(sources)))

print("\nDistribuição por emoção:")
display(pd.Series(y).value_counts().reindex(EMOTIONS, fill_value=0).to_frame("n"))

print("\nDistribuição por dataset:")
display(pd.Series(sources).value_counts().to_frame("n"))

if X_manual.shape[0] == 0 or X_ssl_aligned.shape[0] == 0 or len(y) == 0:
    raise RuntimeError("Ficaste com 0 linhas. Não avances para o treino.")

if len(np.unique(groups)) < 2:
    raise RuntimeError("Há menos de 2 speakers/grupos. Não dá para GroupKFold.")

## 5. Definir modelos e avaliação por speaker

A avaliação principal usa `StratifiedGroupKFold`, para evitar que o mesmo speaker apareça simultaneamente em treino e teste.

In [ ]:
def make_lr(C=1.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=C,
            max_iter=4000,
            class_weight="balanced",
            solver="lbfgs",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])

def make_extra_trees(n_estimators=300):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=n_estimators,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            min_samples_leaf=2,
            max_features="sqrt",
        )),
    ])

def make_rf(n_estimators=250):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            min_samples_leaf=2,
            max_features="sqrt",
        )),
    ])

MODEL_CANDIDATES = {
    "lr_C0.3": make_lr(0.3),
    "lr_C1.0": make_lr(1.0),
}

if RUN_EXTRA_TREES:
    MODEL_CANDIDATES["extra_trees"] = make_extra_trees(EXTRA_TREES_N_ESTIMATORS)

print("Modelos a testar:", list(MODEL_CANDIDATES.keys()))

def normalize_proba(p):
    p = np.asarray(p, dtype=np.float64)
    p = np.nan_to_num(p, nan=1 / len(EMOTIONS), posinf=1 / len(EMOTIONS), neginf=1 / len(EMOTIONS))
    p = np.maximum(p, 1e-9)
    return p / p.sum(axis=1, keepdims=True)

def get_pipeline_classes(fitted_model):
    if hasattr(fitted_model, "classes_"):
        return list(fitted_model.classes_)

    if hasattr(fitted_model, "named_steps"):
        last_step = list(fitted_model.named_steps.values())[-1]
        if hasattr(last_step, "classes_"):
            return list(last_step.classes_)

    raise RuntimeError("Não consegui encontrar classes_ no modelo treinado.")

def predict_proba_aligned(fitted_model, X_test, emotions=EMOTIONS):
    """
    Garante que as probabilidades ficam sempre na ordem EMOTIONS.
    Funciona mesmo se algum fold não tiver uma classe no treino.
    """
    if hasattr(fitted_model, "predict_proba"):
        proba_raw = fitted_model.predict_proba(X_test)
        model_classes = get_pipeline_classes(fitted_model)

        out = np.zeros((X_test.shape[0], len(emotions)), dtype=np.float32)

        for j, cls in enumerate(model_classes):
            cls = str(cls)
            if cls in emotions:
                out[:, emotions.index(cls)] = proba_raw[:, j]

        return normalize_proba(out)

    pred = fitted_model.predict(X_test)
    out = np.full((X_test.shape[0], len(emotions)), 1e-6, dtype=np.float32)

    for i, cls in enumerate(pred):
        cls = str(cls)
        if cls in emotions:
            out[i, emotions.index(cls)] = 1.0

    return normalize_proba(out)

def collapse_penalty_from_pred(pred):
    counts = pd.Series(pred).value_counts(normalize=True)
    max_ratio = float(counts.max()) if len(counts) else 0.0
    return max(0.0, max_ratio - 0.45), max_ratio

def make_cv(y, groups):
    n_groups = len(np.unique(groups))
    n_splits = min(N_SPLITS_GROUP_CV, n_groups)

    if n_splits < 2:
        raise ValueError(f"Não há grupos suficientes para CV: n_groups={n_groups}")

    try:
        return StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE
        ), n_splits
    except Exception:
        return GroupKFold(n_splits=n_splits), n_splits

def evaluate_cv(model_name, model, X, y, groups, feature_set):
    y = np.asarray(y).astype(str)
    groups = np.asarray(groups).astype(str)

    cv, n_splits = make_cv(y, groups)

    oof_pred = np.empty(len(y), dtype=object)
    oof_proba = np.zeros((len(y), len(EMOTIONS)), dtype=np.float32)

    print(f"\n--- {feature_set} / {model_name} ---")

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        clf = clone(model)

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf.fit(X_train, y_train)

        pred = clf.predict(X_test).astype(str)
        proba = predict_proba_aligned(clf, X_test, EMOTIONS)

        oof_pred[test_idx] = pred
        oof_proba[test_idx] = proba

        print(
            f"fold {fold}/{n_splits} | "
            f"test={len(test_idx)} | "
            f"acc={accuracy_score(y_test, pred):.4f} | "
            f"f1_macro={f1_score(y_test, pred, average='macro', zero_division=0):.4f}"
        )

    acc = accuracy_score(y, oof_pred)
    bal_acc = balanced_accuracy_score(y, oof_pred)
    f1_macro = f1_score(y, oof_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y, oof_pred, average="weighted", zero_division=0)
    penalty, max_pred_ratio = collapse_penalty_from_pred(oof_pred)
    score_adjusted = f1_macro - penalty

    return {
        "feature_set": feature_set,
        "model_name": model_name,
        "n_samples": len(y),
        "n_groups": len(np.unique(groups)),
        "n_splits": n_splits,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "max_pred_ratio": max_pred_ratio,
        "collapse_penalty": penalty,
        "score_adjusted": score_adjusted,
        "oof_pred": oof_pred,
        "oof_proba": oof_proba,
    }

def leave_one_source_out(model_name, model, X, y, sources, feature_set):
    y = np.asarray(y).astype(str)
    sources = np.asarray(sources).astype(str)

    rows = []

    for heldout_source in sorted(np.unique(sources)):
        train_idx = np.where(sources != heldout_source)[0]
        test_idx = np.where(sources == heldout_source)[0]

        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        clf = clone(model)
        clf.fit(X[train_idx], y[train_idx])
        pred = clf.predict(X[test_idx]).astype(str)

        rows.append({
            "feature_set": feature_set,
            "model_name": model_name,
            "heldout_source": heldout_source,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "accuracy": accuracy_score(y[test_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
            "f1_macro": f1_score(y[test_idx], pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y[test_idx], pred, average="weighted", zero_division=0),
            "pred_distribution": dict(pd.Series(pred).value_counts()),
        })

    return pd.DataFrame(rows)

## 6. Correr benchmark

Isto é a parte que te dá a qualidade do modelo.

In [ ]:
feature_matrices = {
    "manual": X_manual,
    "ssl_wavlm": X_ssl_aligned,
    "manual_plus_ssl": np.hstack([X_manual, X_ssl_aligned]).astype(np.float32),
}

print("Feature sets:")
for k, v in feature_matrices.items():
    print(k, v.shape)

results = []
oof_store = {}
source_tables = []

t_global = time.time()

for feature_set, X in feature_matrices.items():
    print("\n" + "=" * 90)
    print("FEATURE SET:", feature_set, X.shape)
    print("=" * 90)

    for model_name, model in MODEL_CANDIDATES.items():
        t0 = time.time()

        try:
            res = evaluate_cv(model_name, model, X, y, groups, feature_set)
            row = {k: v for k, v in res.items() if k not in {"oof_pred", "oof_proba"}}
            row["elapsed_sec"] = time.time() - t0

            results.append(row)
            oof_store[(feature_set, model_name)] = {
                "pred": res["oof_pred"],
                "proba": res["oof_proba"],
            }

            if RUN_LEAVE_ONE_SOURCE_OUT and model_name.startswith("lr"):
                source_tables.append(
                    leave_one_source_out(model_name, model, X, y, sources, feature_set)
                )

        except Exception as e:
            print("❌ Erro:", feature_set, model_name, repr(e))

if not results:
    raise RuntimeError("Nenhum modelo conseguiu ser avaliado.")

benchmark_df = pd.DataFrame(results).sort_values("score_adjusted", ascending=False)
benchmark_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME.csv"
benchmark_df.to_csv(benchmark_path, index=False)

print("\n✅ Benchmark concluído em", round(time.time() - t_global, 1), "segundos")
print("Guardado em:", benchmark_path)
display(benchmark_df)

if source_tables:
    source_eval_df = pd.concat(source_tables, ignore_index=True)
    source_path = OUTPUT_DIR / "leave_one_source_out_v5_RESUME.csv"
    source_eval_df.to_csv(source_path, index=False)
    print("Leave-One-Source-Out guardado em:", source_path)
    display(source_eval_df.sort_values(["feature_set", "model_name", "heldout_source"]))

## 7. Ensemble simples manual + SSL

Combina as probabilidades do melhor modelo manual e do melhor modelo SSL. Isto às vezes melhora a estabilidade.

In [ ]:
def weighted_average(probas, weights):
    w = np.asarray(weights, dtype=np.float64)
    w = w / (w.sum() + 1e-12)

    out = np.zeros_like(probas[0], dtype=np.float64)
    for p, wi in zip(probas, w):
        out += wi * p

    return normalize_proba(out)

benchmark_df2 = benchmark_df.copy()
ensemble_info = None

manual_rows = [r for r in results if r["feature_set"] == "manual"]
ssl_rows = [r for r in results if r["feature_set"] == "ssl_wavlm"]

if manual_rows and ssl_rows:
    best_manual = sorted(manual_rows, key=lambda r: r["score_adjusted"], reverse=True)[0]
    best_ssl = sorted(ssl_rows, key=lambda r: r["score_adjusted"], reverse=True)[0]

    key_manual = (best_manual["feature_set"], best_manual["model_name"])
    key_ssl = (best_ssl["feature_set"], best_ssl["model_name"])

    wm = max(0.01, best_manual["f1_macro"])
    ws = max(0.01, best_ssl["f1_macro"])

    p_ens = weighted_average(
        [oof_store[key_manual]["proba"], oof_store[key_ssl]["proba"]],
        [wm, ws]
    )

    pred_ens_idx = p_ens.argmax(axis=1)
    pred_ens = np.array([EMOTIONS[i] for i in pred_ens_idx], dtype=object)

    penalty, max_pred_ratio = collapse_penalty_from_pred(pred_ens)

    ens = {
        "feature_set": "ensemble_manual_ssl",
        "model_name": f"{key_manual[1]}+{key_ssl[1]}",
        "n_samples": len(y),
        "n_groups": len(np.unique(groups)),
        "n_splits": benchmark_df["n_splits"].max(),
        "accuracy": accuracy_score(y, pred_ens),
        "balanced_accuracy": balanced_accuracy_score(y, pred_ens),
        "f1_macro": f1_score(y, pred_ens, average="macro", zero_division=0),
        "f1_weighted": f1_score(y, pred_ens, average="weighted", zero_division=0),
        "max_pred_ratio": max_pred_ratio,
        "collapse_penalty": penalty,
        "manual_weight": wm / (wm + ws),
        "ssl_weight": ws / (wm + ws),
        "elapsed_sec": 0.0,
    }

    ens["score_adjusted"] = ens["f1_macro"] - ens["collapse_penalty"]

    ensemble_info = {
        "best_manual": best_manual,
        "best_ssl": best_ssl,
        "ensemble": ens,
        "pred": pred_ens,
        "proba": p_ens,
    }

    benchmark_df2 = pd.concat(
        [benchmark_df, pd.DataFrame([ens])],
        ignore_index=True
    ).sort_values("score_adjusted", ascending=False)

    ensemble_path = OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_with_ensemble.csv"
    benchmark_df2.to_csv(ensemble_path, index=False)

    print("✅ Ensemble calculado")
    print("Pesos:", {"manual": ens["manual_weight"], "ssl": ens["ssl_weight"]})
    print("Guardado em:", ensemble_path)
    display(benchmark_df2)

else:
    print("Ensemble não disponível: falta ramo manual ou SSL.")

## 8. Melhor resultado, relatório por classe e matriz de confusão

In [ ]:
best = benchmark_df2.iloc[0].to_dict()
best_key = (best["feature_set"], best["model_name"])

print("🏆 Melhor resultado:")
print(best)

if best["feature_set"] == "ensemble_manual_ssl" and ensemble_info is not None:
    best_pred = ensemble_info["pred"]
    best_proba = ensemble_info["proba"]
else:
    best_pred = oof_store[best_key]["pred"]
    best_proba = oof_store[best_key]["proba"]

report_text = classification_report(
    y,
    best_pred,
    labels=EMOTIONS,
    target_names=EMOTIONS,
    digits=4,
    zero_division=0,
)

print("\nClassification report:")
print(report_text)

report_path = OUTPUT_DIR / "best_classification_report_v5_RESUME.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("Best model:\n")
    f.write(json.dumps({k: str(v) for k, v in best.items()}, ensure_ascii=False, indent=2))
    f.write("\n\nClassification report:\n")
    f.write(report_text)

print("Relatório guardado em:", report_path)

cm = confusion_matrix(y, best_pred, labels=EMOTIONS)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=EMOTIONS)
disp.plot(ax=ax, xticks_rotation=35, values_format="d", colorbar=True)
ax.set_title(f"Matriz de confusão — {best_key[0]} / {best_key[1]}")
plt.tight_layout()

cm_path = OUTPUT_DIR / "best_confusion_matrix_v5_RESUME.png"
plt.savefig(cm_path, dpi=170)
plt.show()

print("Matriz de confusão guardada em:", cm_path)

pred_counts = pd.Series(best_pred).value_counts().reindex(EMOTIONS, fill_value=0)

plt.figure(figsize=(9, 4))
plt.bar(pred_counts.index, pred_counts.values)
plt.title(f"Distribuição de previsões OOF — {best_key[0]} / {best_key[1]}")
plt.ylabel("Nº previsões")
plt.xticks(rotation=35, ha="right")

for i, v in enumerate(pred_counts.values):
    plt.text(i, v, str(int(v)), ha="center", va="bottom")

plt.tight_layout()
dist_path = OUTPUT_DIR / "best_prediction_distribution_v5_RESUME.png"
plt.savefig(dist_path, dpi=170)
plt.show()

print("Distribuição de previsões guardada em:", dist_path)

## 9. Treinar modelo final e guardar artefacto

Este artefacto guarda o classificador final treinado nas features em cache.  
Nota: este notebook não reimplementa extração de features para áudio novo; é para avaliação e benchmark a partir da cache.

In [ ]:
def fit_final_classifier(feature_set, model_name):
    clf = clone(MODEL_CANDIDATES[model_name])
    clf.fit(feature_matrices[feature_set], y)
    return clf

best_final = benchmark_df2.iloc[0].to_dict()
print("Best final:", best_final)

artifact = {
    "version": "prosody_v5_resume_cache_only",
    "emotions": EMOTIONS,
    "class_priors": class_priors,
    "manual_feature_cols": manual_feature_cols,
    "benchmark": benchmark_df2.to_dict(orient="records"),
    "selected_score": best_final,
    "note": "Artefacto cache-only: avalia/treina usando features já extraídas. Não contém extração WavLM para áudio novo.",
}

if best_final["feature_set"] == "ensemble_manual_ssl" and ensemble_info is not None:
    bm = ensemble_info["best_manual"]
    bs = ensemble_info["best_ssl"]
    ens = ensemble_info["ensemble"]

    artifact.update({
        "selected_strategy": "ensemble_manual_ssl",
        "manual_model_name": bm["model_name"],
        "ssl_model_name_clf": bs["model_name"],
        "manual_model": fit_final_classifier("manual", bm["model_name"]),
        "ssl_model": fit_final_classifier("ssl_wavlm", bs["model_name"]),
        "manual_weight": float(ens["manual_weight"]),
        "ssl_weight": float(ens["ssl_weight"]),
    })
else:
    fs = best_final["feature_set"]
    mn = best_final["model_name"]

    artifact.update({
        "selected_strategy": fs,
        "final_model_name": mn,
        "final_model": fit_final_classifier(fs, mn),
    })

artifact_path = OUTPUT_DIR / "prosody_final_artifact_v5_RESUME_cache_only.joblib"
joblib.dump(artifact, artifact_path)

print("✅ Artefacto guardado em:", artifact_path)
print("Estratégia:", artifact["selected_strategy"])

## 10. Resumo final automático

In [ ]:
summary = {
    "best_feature_set": best_final["feature_set"],
    "best_model_name": best_final["model_name"],
    "accuracy": round(float(best_final["accuracy"]), 4),
    "balanced_accuracy": round(float(best_final["balanced_accuracy"]), 4),
    "f1_macro": round(float(best_final["f1_macro"]), 4),
    "f1_weighted": round(float(best_final["f1_weighted"]), 4),
    "score_adjusted": round(float(best_final["score_adjusted"]), 4),
    "n_samples": int(best_final["n_samples"]),
    "n_groups": int(best_final["n_groups"]),
}

summary_path = OUTPUT_DIR / "summary_v5_RESUME.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Resumo:")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nGuardado em:", summary_path)

print("\nFicheiros principais gerados:")
for p in [
    OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME.csv",
    OUTPUT_DIR / "prosody_model_benchmark_v5_RESUME_with_ensemble.csv",
    OUTPUT_DIR / "best_classification_report_v5_RESUME.txt",
    OUTPUT_DIR / "best_confusion_matrix_v5_RESUME.png",
    OUTPUT_DIR / "best_prediction_distribution_v5_RESUME.png",
    OUTPUT_DIR / "prosody_final_artifact_v5_RESUME_cache_only.joblib",
    OUTPUT_DIR / "summary_v5_RESUME.json",
]:
    print("-", p.name, "->", p.exists())